# Day 3 Lab - CPU Spike Simulator & Elastic Cloud Uploader
### AIOps Fundamentals Training | Microland

---

## What This Notebook Does

This notebook generates **time-series CPU metric data** that simulates what Metricbeat would
ship from a real Windows or Linux server. The data covers a **6-hour window** and contains:

- **5.5 hours** of realistic, fluctuating baseline CPU usage (normal operational traffic)
- **30 minutes** of an injected CPU spike at **3x the baseline level** - placed at the 3-hour mark

Once uploaded to Elastic Cloud, you will create an ML Anomaly Detection job in Kibana
and test whether it detects your injected spike - and what anomaly score it assigns.

---

## Learning Objectives
- Understand how time-series metric data is structured for ML consumption
- See how a baseline is established from stable data
- Observe how a 3x spike appears both visually and as an ML anomaly score
- Understand the difference between a deterministic threshold alert and a probabilistic ML score

---

## Before You Start
1. Day 1 lab complete - you have a running Elastic Cloud deployment
2. You have your **Cloud ID** and **API Key** ready
3. Running in **Google Colab**

> **No Python knowledge required.** Run each cell in order using the Play button or Shift+Enter.


## Step 1 - Install Required Libraries


In [ ]:
!pip install elasticsearch --quiet
print('Libraries installed.')


## Step 2 - Configuration

Enter your Elastic Cloud credentials below.
You can also adjust the simulation parameters if you want to experiment.


In [ ]:
# -----------------------------------------------------------
# ELASTIC CLOUD CREDENTIALS - fill in your own values
# -----------------------------------------------------------
CLOUD_ID   = 'YOUR_CLOUD_ID_HERE'
API_KEY    = 'YOUR_API_KEY_HERE'
INDEX_NAME = 'aiops-lab-day3'

# -----------------------------------------------------------
# SIMULATION PARAMETERS - feel free to adjust and re-run
# -----------------------------------------------------------
SERVER_NAME        = 'WINTEL-SRV-02'   # The simulated server name
TOTAL_HOURS        = 6                 # Total hours of data to generate
SAMPLE_INTERVAL_S  = 30               # Metricbeat default: one sample every 30 seconds

# Baseline CPU: server idles between 15% and 35% with small random fluctuation
BASELINE_MEAN      = 0.25             # 25% average CPU
BASELINE_NOISE     = 0.05             # +/- 5% natural variation

# Spike configuration
SPIKE_START_HOUR   = 3.0              # Spike begins 3 hours into the window
SPIKE_DURATION_MIN = 30              # Spike lasts 30 minutes
SPIKE_MULTIPLIER   = 3.0             # Spike is 3x the baseline mean

print('Configuration set.')
print(f'   Server         : {SERVER_NAME}')
print(f'   Window         : {TOTAL_HOURS} hours at {SAMPLE_INTERVAL_S}s intervals')
print(f'   Baseline CPU   : {BASELINE_MEAN*100:.0f}% mean, +/-{BASELINE_NOISE*100:.0f}% noise')
print(f'   Spike at       : hour {SPIKE_START_HOUR} for {SPIKE_DURATION_MIN} minutes at {SPIKE_MULTIPLIER}x baseline')
print(f'   Expected spike : ~{BASELINE_MEAN * SPIKE_MULTIPLIER * 100:.0f}% CPU during spike window')


## Step 3 - Connect to Elastic Cloud


In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch(cloud_id=CLOUD_ID, api_key=API_KEY)
info = es.info()
print(f'Connected to cluster: {info["cluster_name"]}')
print(f'   Version: {info["version"]["number"]}')


## Step 4 - Create the Index

The field names below mirror the Metricbeat `system.cpu` schema exactly.
This means the ML job you create in Kibana will use the same field names
as a real production Metricbeat deployment.


In [ ]:
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f'Deleted existing index: {INDEX_NAME}')

mapping = {
    'mappings': {
        'properties': {
            '@timestamp':              {'type': 'date'},
            'host.name':               {'type': 'keyword'},
            'host.os.name':            {'type': 'keyword'},
            'system.cpu.total.pct':    {'type': 'float'},
            'system.cpu.user.pct':     {'type': 'float'},
            'system.cpu.system.pct':   {'type': 'float'},
            'system.cpu.idle.pct':     {'type': 'float'},
            'metricset.name':          {'type': 'keyword'},
            'event.dataset':           {'type': 'keyword'},
            'is_spike':                {'type': 'boolean'},
            'spike_description':       {'type': 'text'}
        }
    }
}

es.indices.create(index=INDEX_NAME, body=mapping)
print(f'Index "{INDEX_NAME}" created with Metricbeat-compatible CPU mapping.')


## Step 5 - Generate the CPU Time-Series Data

This cell creates one metric sample every 30 seconds for the full 6-hour window.
That gives us **720 data points** - a realistic density for Metricbeat collection.

### How the Spike Is Modelled

The baseline CPU follows a realistic pattern:
- A slow sinusoidal drift (simulating workload cycles across the day)
- Random Gaussian noise (simulating natural process variation)
- Occasional micro-bursts (simulating brief process spikes)

The injected spike multiplies the mean by 3x and adds higher noise,
mimicking a runaway process consuming CPU aggressively.

> **AIOps Concept:** The ML model will learn the baseline pattern from the stable 5.5 hours
> and flag the 30-minute spike because it deviates significantly from what it learned as normal.


In [ ]:
import random
import math
import json
from datetime import datetime, timedelta, timezone

random.seed(2024)

now        = datetime.now(timezone.utc)
start_time = now - timedelta(hours=TOTAL_HOURS)

spike_start = start_time + timedelta(hours=SPIKE_START_HOUR)
spike_end   = spike_start + timedelta(minutes=SPIKE_DURATION_MIN)

total_samples = int((TOTAL_HOURS * 3600) / SAMPLE_INTERVAL_S)

events = []

for i in range(total_samples):
    ts = start_time + timedelta(seconds=i * SAMPLE_INTERVAL_S)
    in_spike = spike_start <= ts <= spike_end

    # Sinusoidal drift across the window (slow workload cycle)
    cycle = math.sin(2 * math.pi * i / total_samples) * 0.04

    if in_spike:
        # Spike: mean is SPIKE_MULTIPLIER * baseline, higher noise
        cpu_mean   = BASELINE_MEAN * SPIKE_MULTIPLIER
        cpu_noise  = BASELINE_NOISE * 2.5
        cpu_total  = min(0.98, max(0.05, cpu_mean + cycle + random.gauss(0, cpu_noise)))
        is_spike   = True
        spike_desc = (
            f'Injected CPU spike: {SPIKE_MULTIPLIER}x baseline '
            f'({SPIKE_START_HOUR}h mark, {SPIKE_DURATION_MIN} min duration). '
            f'Simulates a runaway process consuming CPU.'
        )
    else:
        # Baseline: normal operating range with occasional micro-bursts
        micro_burst = 0.08 if random.random() < 0.03 else 0.0  # 3% chance
        cpu_total   = min(0.75, max(0.05,
                          BASELINE_MEAN + cycle + micro_burst
                          + random.gauss(0, BASELINE_NOISE)))
        is_spike   = False
        spike_desc = ''

    # Decompose total into user/system/idle (realistic split)
    user_frac   = random.uniform(0.55, 0.70)
    system_frac = random.uniform(0.15, 0.25)
    idle_frac   = max(0.0, 1.0 - cpu_total)

    events.append({
        '@timestamp':            ts.isoformat(),
        'host.name':             SERVER_NAME,
        'host.os.name':          'Windows Server 2019',
        'system.cpu.total.pct':  round(cpu_total, 4),
        'system.cpu.user.pct':   round(cpu_total * user_frac, 4),
        'system.cpu.system.pct': round(cpu_total * system_frac, 4),
        'system.cpu.idle.pct':   round(idle_frac, 4),
        'metricset.name':        'cpu',
        'event.dataset':         'system.cpu',
        'is_spike':              is_spike,
        'spike_description':     spike_desc
    })

spike_points   = [e for e in events if     e['is_spike']]
normal_points  = [e for e in events if not e['is_spike']]
avg_baseline   = sum(e['system.cpu.total.pct'] for e in normal_points)  / len(normal_points)
avg_spike      = sum(e['system.cpu.total.pct'] for e in spike_points)   / len(spike_points)
peak_spike     = max(e['system.cpu.total.pct'] for e in spike_points)

print(f'Generated {len(events):,} CPU metric samples')
print(f'   Sample interval   : every {SAMPLE_INTERVAL_S}s')
print(f'   Baseline samples  : {len(normal_points):,}')
print(f'   Spike samples     : {len(spike_points):,}  ({SPIKE_DURATION_MIN} minutes)')
print()
print(f'CPU Statistics:')
print(f'   Baseline average  : {avg_baseline*100:.1f}%')
print(f'   Spike average     : {avg_spike*100:.1f}%  ({avg_spike/avg_baseline:.1f}x baseline)')
print(f'   Spike peak        : {peak_spike*100:.1f}%')
print(f'   Spike window      : {spike_start.strftime("%H:%M")} to {spike_end.strftime("%H:%M")} UTC')


## Step 6 - Visualise the Data Locally (Optional but Recommended)

Before uploading, plot the CPU time series here in the notebook.
This gives you a clear picture of what the ML job will see.

> **Observation:** The spike should be visually obvious in this plot.
> However, remember that in a real environment this metric is one of thousands.
> Without ML, an engineer would need to manually check every server's CPU chart to find it.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime

timestamps = [datetime.fromisoformat(e['@timestamp'].replace('Z', '+00:00')) for e in events]
cpu_values = [e['system.cpu.total.pct'] * 100 for e in events]
colors     = ['#C55A11' if e['is_spike'] else '#1B3A6B' for e in events]

fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(timestamps, cpu_values, c=colors, s=4, alpha=0.7, zorder=2)
ax.axvline(spike_start, color='#C55A11', linestyle='--', linewidth=1.5, alpha=0.8, label='Spike start')
ax.axvline(spike_end,   color='#C55A11', linestyle='--', linewidth=1.5, alpha=0.8, label='Spike end')
ax.axhspan(spike_start, spike_end, alpha=0)  # placeholder
ax.axvspan(spike_start, spike_end, alpha=0.08, color='#C55A11', label='Spike window')
ax.axhline(avg_baseline * 100, color='#0D6E6E', linestyle=':', linewidth=1.5,
           label=f'Baseline avg ({avg_baseline*100:.1f}%)')

baseline_patch = mpatches.Patch(color='#1B3A6B', label='Baseline CPU (normal)')
spike_patch    = mpatches.Patch(color='#C55A11', label=f'Spike CPU ({SPIKE_MULTIPLIER}x baseline)')

ax.set_xlabel('Time (UTC)', fontsize=11)
ax.set_ylabel('CPU Total %', fontsize=11)
ax.set_title(f'Simulated CPU Time Series - {SERVER_NAME}\n'
             f'6 hours of data | 30-minute spike at 3x baseline injected at hour 3',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax.legend(handles=[baseline_patch, spike_patch], loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print('Plot rendered. The orange region is the 30-minute injected spike.')
print('This is what the Elastic ML job will need to detect automatically.')


## Step 7 - Upload to Elastic Cloud

Uploading all 720 CPU metric samples via the bulk API.


In [ ]:
from elasticsearch.helpers import bulk

def generate_actions(docs, index):
    for doc in docs:
        yield {'_index': index, '_source': doc}

print(f'Uploading {len(events):,} metric samples to index "{INDEX_NAME}" ...')

success, errors = bulk(
    es,
    generate_actions(events, INDEX_NAME),
    chunk_size=200,
    raise_on_error=False
)

print(f'Upload complete!')
print(f'   Documents indexed : {success:,}')
if errors:
    print(f'   Errors            : {len(errors)}')
    for e in errors[:3]: print(f'      {e}')
else:
    print(f'   Errors            : 0')

es.indices.refresh(index=INDEX_NAME)
print('Index refreshed - data is now searchable in Kibana.')


## Step 8 - Verify the Upload


In [ ]:
total = es.count(index=INDEX_NAME)['count']
spike_count = es.count(index=INDEX_NAME,
    body={'query': {'term': {'is_spike': True}}})['count']

stats = es.search(index=INDEX_NAME, body={
    'size': 0,
    'aggs': {
        'avg_cpu':  {'avg':  {'field': 'system.cpu.total.pct'}},
        'max_cpu':  {'max':  {'field': 'system.cpu.total.pct'}},
        'min_cpu':  {'min':  {'field': 'system.cpu.total.pct'}},
    }
})
aggs = stats['aggregations']

print(f'Verification summary - index "{INDEX_NAME}":')
print(f'   Total samples   : {total:,}')
print(f'   Spike samples   : {spike_count:,}  (30-min window)')
print(f'   Normal samples  : {total - spike_count:,}')
print()
print(f'CPU statistics across all samples:')
print(f'   Average : {aggs["avg_cpu"]["value"]*100:.1f}%')
print(f'   Maximum : {aggs["max_cpu"]["value"]*100:.1f}%  <- spike peak')
print(f'   Minimum : {aggs["min_cpu"]["value"]*100:.1f}%')


## Step 9 - Create the ML Anomaly Detection Job in Kibana

Now that your data is in Elastic Cloud, follow these steps to create an ML job
that will detect the injected spike automatically.

---

### 9a - Create the Anomaly Detection Job

1. In Kibana, go to: **Machine Learning > Anomaly Detection > Create Job**
2. Select index: `aiops-lab-day3`
3. Choose job type: **Single Metric**
4. Configure the job:

| Setting | Value |
|---------|-------|
| Function | `high_mean` |
| Field | `system.cpu.total.pct` |
| Bucket span | `15m` |
| Job ID | `day3-cpu-spike-detection` |

5. Set the time range to **Full** (to cover all 6 hours of data)
6. Click **Create Job** and wait for it to complete

---

### 9b - Analyse the Results in Anomaly Explorer

1. Go to **Machine Learning > Anomaly Explorer**
2. Select your job: `day3-cpu-spike-detection`
3. Look for the swimlane heatmap - you should see a red/orange block at the 3-hour mark
4. Click on the anomaly block and note:
   - The **anomaly score** (should be 75+)
   - The **actual value** (the CPU% during the spike)
   - The **typical value** (what the ML expected to see at that time)

---

### 9c - Create a Forecast

1. Go to **Machine Learning > Single Metric Viewer**
2. Select your job
3. Click **Forecast** > set duration to **7 days**
4. Observe:
   - The projected trend line
   - The confidence interval bands (the shaded area around the forecast)

---

### Discussion Questions

> **Q1:** What anomaly score did the ML assign to the spike? Look at the score table below
> and describe what it means in plain English.

| Score | Meaning |
|-------|---------|
| 0-24 | Normal - within expected range |
| 25-49 | Slight deviation - monitor |
| 50-74 | Notable - investigate |
| 75-100 | High anomaly - statistically very unusual |

> **Q2:** The spike in this dataset was 3x the baseline. If it had been 1.5x the baseline,
> would the ML still detect it? What would the score likely be?

> **Q3:** Is the ML alert produced here deterministic or probabilistic? How do you know?

---

### Lab Complete

You have:
- Generated 720 realistic CPU metric data points with a hidden 30-minute spike
- Visualised the spike and understood how it differs from the baseline
- Uploaded the data to Elastic Cloud
- Created an ML anomaly detection job
- Observed the anomaly score and forecast output

Move to **Day 4** where you will apply Log Rate Analysis to find the root cause
of a simulated multi-event incident.


## Optional - Experiment: Change the Spike Multiplier

Want to see how the ML score changes with a smaller or larger spike?
Change `NEW_MULTIPLIER` below and re-run this cell to upload a fresh dataset.

Try values like `1.5`, `2.0`, `4.0` and observe whether the ML detects them
and what score they receive.


In [ ]:
# Change this value and re-run to see how anomaly score changes
NEW_MULTIPLIER = 2.0

print(f'Generating dataset with {NEW_MULTIPLIER}x spike multiplier...')

events_exp = []
for i in range(total_samples):
    ts = start_time + timedelta(seconds=i * SAMPLE_INTERVAL_S)
    in_spike = spike_start <= ts <= spike_end
    cycle = math.sin(2 * math.pi * i / total_samples) * 0.04

    if in_spike:
        cpu_mean  = BASELINE_MEAN * NEW_MULTIPLIER
        cpu_noise = BASELINE_NOISE * 2.0
        cpu_total = min(0.98, max(0.05, cpu_mean + cycle + random.gauss(0, cpu_noise)))
        is_spike  = True
    else:
        micro_burst = 0.08 if random.random() < 0.03 else 0.0
        cpu_total   = min(0.75, max(0.05,
                          BASELINE_MEAN + cycle + micro_burst + random.gauss(0, BASELINE_NOISE)))
        is_spike = False

    user_frac   = random.uniform(0.55, 0.70)
    system_frac = random.uniform(0.15, 0.25)

    events_exp.append({
        '@timestamp':            ts.isoformat(),
        'host.name':             f'{SERVER_NAME}-EXP',
        'host.os.name':          'Windows Server 2019',
        'system.cpu.total.pct':  round(cpu_total, 4),
        'system.cpu.user.pct':   round(cpu_total * user_frac, 4),
        'system.cpu.system.pct': round(cpu_total * system_frac, 4),
        'system.cpu.idle.pct':   round(max(0.0, 1.0 - cpu_total), 4),
        'metricset.name':        'cpu',
        'event.dataset':         'system.cpu',
        'is_spike':              is_spike,
        'spike_description':     f'{NEW_MULTIPLIER}x spike experiment'
    })

# Upload to a separate index so it does not overwrite the main lab data
EXP_INDEX = f'aiops-lab-day3-exp-{str(NEW_MULTIPLIER).replace(".","p")}x'
if es.indices.exists(index=EXP_INDEX):
    es.indices.delete(index=EXP_INDEX)

es.indices.create(index=EXP_INDEX, body=mapping)

success, _ = bulk(es, generate_actions(events_exp, EXP_INDEX),
                  chunk_size=200, raise_on_error=False)
es.indices.refresh(index=EXP_INDEX)

avg_b = sum(e['system.cpu.total.pct'] for e in events_exp if not e['is_spike']) / \
        sum(1 for e in events_exp if not e['is_spike'])
avg_s = sum(e['system.cpu.total.pct'] for e in events_exp if     e['is_spike']) / \
        sum(1 for e in events_exp if e['is_spike'])

print(f'Experiment dataset uploaded to index: {EXP_INDEX}')
print(f'   Baseline avg CPU : {avg_b*100:.1f}%')
print(f'   Spike avg CPU    : {avg_s*100:.1f}%  ({avg_s/avg_b:.1f}x baseline)')
print()
print('Create a new ML job pointing to this index and compare the anomaly score.')
print(f'In Kibana, use index pattern: {EXP_INDEX}')
